# Data Analysis for Bacillus subtilis Biofilms

## Data Analysis for Biofilm Snapshots

### Select a Folder with .csv Files

In [ ]:
import ipywidgets as widgets
from IPython.display import display

path_text = widgets.Textarea(
    value='',
    placeholder='Paste or type folder path here',
    description='Folder:',
    disabled=False,
    layout=widgets.Layout(width='600px')
)

button = widgets.Button(description="Confirm")

def on_button_click(b):
    global input_dir
    input_dir = path_text.value.strip()
    print("Selected folder:", input_dir)

button.on_click(on_button_click)

display(path_text, button)

### Grpahs plotting normalised fluorescence profiles, as well as FM-ratios for every individual biofilm

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({'font.size': 30, 'font.family': 'Mathematica1', 'lines.linewidth': 4})

# Validate path
if "input_dir" not in globals() or not os.path.isdir(input_dir):
    raise SystemExit("No valid folder selected. Run the first cell and click Confirm.")

# Collect all CSV files from subfolders
all_files = []
for root, dirs, files in os.walk(input_dir):
    for f in files:
        if f.endswith(".csv"):
            all_files.append(os.path.join(root, f))

# Filter for mCherry and YFP CSV files
mcherry_files = [f for f in all_files if f.endswith("_RadialProfile_Angle_mCherry.csv")]
yfp_files = [f for f in all_files if f.endswith("_RadialProfile_Angle_YFP.csv")]
#cfp_files = [f for f in all_files if f.endswith("_RadialProfile_Angle_CFP.csv")]
threshold_files = [f for f in all_files if f.endswith("_RadialProfile_Angle_Threshold.csv")]

# Ensure matching files by base name
for mcherry_file in mcherry_files:
    base = os.path.basename(mcherry_file).replace("_RadialProfile_Angle_mCherry.csv", "")
    yfp_file = next((f for f in yfp_files if os.path.basename(f) == f"{base}_RadialProfile_Angle_YFP.csv"), None)
    #cfp_file = next((f for f in cfp_files if os.path.basename(f) == f"{base}_RadialProfile_Angle_CFP.csv"), None)
    threshold_file = next((f for f in threshold_files if os.path.basename(f) == f"{base}_RadialProfile_Angle_Threshold.csv"), None)
    if not yfp_file:
        print(f"Warning: file not found for {base}, skipping.")
        continue
    #if not cfp_file:
    #    print(f"Warning: CFP file not found for {base}, skipping.")
    #    continue
    if not threshold_file:
        print(f"Warning: threshold file not found for {base}, skipping.")
        continue  # added missing continue for threshold

    # Use the subfolder containing the current files
    subfolder = os.path.dirname(mcherry_file)
    output_dir = os.path.join(subfolder, "Plots")
    os.makedirs(output_dir, exist_ok=True)

    
    #nread csv
    mcherry_df = pd.read_csv(mcherry_file).dropna(how='all').reset_index(drop=True)
    yfp_df = pd.read_csv(yfp_file).dropna(how='all').reset_index(drop=True)
    #cfp_df = pd.read_csv(cfp_file).dropna(how='all').reset_index(drop=True)
    threshold_df = pd.read_csv(threshold_file).dropna(how='all').reset_index(drop=True)

    #check lengths
    if not (len(mcherry_df) == len(yfp_df) == len(threshold_df)):
        print(f"Length mismatch AFTER cleaning for {base}, skipping.")
        continue

    #Eextract the columns
    x = mcherry_df["Distance"]
    mcherry_y = mcherry_df["Intensity"]
    yfp_y = yfp_df["Intensity"]
    #cfp_y = cfp_df["Intensity"]
    threshold_y = threshold_df["Intensity"]

   

  # edge detection that takes multiple threshold crossings into account and uses the last as an edge

    # Find the last index where the value is 255
    idx_255 = threshold_y[threshold_y == 255].index

    if len(idx_255) == 0:
         # No 255 at all → fallback to last sample
        cutoff_index = len(threshold_y) - 1
    else:
        last_255_index = idx_255[-1]
            
        # Find indices AFTER the last 255 where value drops below 15
        drops_after_255 = threshold_y[
            (threshold_y < 15) & (threshold_y.index > last_255_index)
        ].index

        if len(drops_after_255) > 0:
            # First drop after the final 255 = the edge
            cutoff_index = drops_after_255[0]
        else:
            # Never dropped after last 255
            cutoff_index = len(threshold_y) - 1

    #Get x value of the edge
    cutoff_index = max(0, min(cutoff_index, len(x) - 1))
    cutoff_x = x.iloc[cutoff_index]
   

    # Cut off the data at the colony edge
    x_trunc = x[x <= cutoff_x]
    mcherry_trunc = mcherry_y.iloc[:len(x_trunc)]
    yfp_trunc = yfp_y.iloc[:len(x_trunc)]
    #cfp_trunc = cfp_y.iloc[:len(x_trunc)]

    # Normalize x-axis, the edge of the colony is 1
    x_norm = x_trunc / x_trunc.iloc[-1]
    
    
    #calculate the matrix/motillity ratio
    ratio = mcherry_trunc / yfp_trunc.replace(0, pd.NA)  # Avoid divide by zero
    ratio = ratio.fillna(0)  # handle division NaNs safely
    max_ratio = ratio.max()
    position = ratio.idxmax()  # fixed: index() → idxmax()
    max_radius = x_norm.iloc[position]
    
    
    #calculate the motillity/matrix ratio
    ratio_2 = yfp_trunc.replace(0, pd.NA) / mcherry_trunc.replace(0, pd.NA)# Avoid divide by zero
    ratio_2 = ratio_2.fillna(0)  # handle division NaNs safely
    max_ratio_2 = ratio_2.max()
    position_2 = ratio_2.idxmax()  # fixed: index() → idxmax()
    max_radius_2 = x_norm.iloc[position_2]

    #normalise the flagellated signal to the constitutive signal
    #yfp_normalized = yfp_trunc / cfp_trunc.replace(0, pd.NA)  # Avoid divide by zero
    #yfp_normalized = yfp_normalized.fillna(0)  # handle division NaNs safely
    #max_normalized = yfp_normalized.max()
    #position_normalized = yfp_normalized.idxmax()  # fixed: index() → idxmax()
    #max_radius_normalized = x_norm.iloc[position_normalized]
    

    # Plot mCherry
    plt.figure(figsize=(9,6))
    plt.plot(x_norm, mcherry_trunc, label='Matrix', color='red')
    plt.xlabel("$r_{norm}$")
    plt.ylabel("Intensity (a.u.)")
    #plt.title(f"{base} - Matrix")
    #plt.legend()
    plt.xlim(0, 1)  # set x-axis limit to show full range of $r_{norm}$
    plt.ylim(0,75)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f"{base}_Matrix_Normalized.pdf"))
    plt.close()

    # Plot YFP
    plt.figure(figsize=(9,6))
    plt.plot(x_norm, yfp_trunc, label='Flagellated', color='orange')
    plt.xlabel("$r_{norm}$")
    plt.ylabel("Intensity (a.u.)")
    plt.xlim(0, 1)  # set x-axis limit to show full range of $r_{norm}$
    plt.ylim(0,75)
    #plt.title(f"{base} - Motility")
    #plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f"{base}_Motility_Normalized.pdf"))
    plt.close()

    # Plot CFP
    #plt.figure(figsize=(6,4))
    #plt.plot(x_norm, cfp_trunc, label='Constitutive', color='blue')
    #plt.xlabel("$r_{norm}$")
    #plt.ylabel("Intensity (a.u.)")
    #plt.title(f"{base} - CFP")
    #plt.legend()
    #plt.tight_layout()
    #plt.savefig(os.path.join(output_dir, f"{base}_CFP_Normalized.pdf"))
    #plt.close()
    
    # Plot mCherry and YFP in the same graph
    plt.figure(figsize=(9,6))
    plt.plot(x_norm, mcherry_trunc, label='Matrix', color='red')
    plt.plot(x_norm, yfp_trunc, label='Flagellated', color='orange')
    #plt.plot(x_norm, cfp_trunc, label='Constitutive', color='blue')
    plt.xlabel("$r_{norm}$")
    plt.ylabel("$Intensity \ (a.u.)$")
    plt.xlim(0, 1)  # set x-axis limit to show full range of $r_{norm}$
    plt.ylim(0,110)  # set y-axis limit to start at 0
    #plt.title(f"{base}")
    #plt.legend(loc='upper left', bbox_to_anchor=(1, 1), fontsize='small')
    plt.legend(loc='upper left', fontsize='small')
    #plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f"{base}_Normalized.pdf"))
    plt.close()
    
    # Plot mCherry, YFP and threshold data in the same graph with the original radius
    plt.figure(figsize=(9,6))
    plt.plot(x, mcherry_y, label='Matrix', color='red')
    plt.plot(x, yfp_y, label='Flagellated', color='orange')
    #plt.plot(x, cfp_y, label='Constitutive', color='blue')
    plt.plot(x, threshold_y, label='Mask', color = 'black')
    plt.xlabel("$Radius \ (pxl)$")
    plt.ylabel("$Intensity \ (a.u.)$")
    #plt.title(f"{base}")
    plt.legend(loc='upper left', fontsize='small')
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f"{base}_non_Normalized.pdf"))
    plt.close()

    # Plot ratio mCherry / YFP
    #plt.figure(figsize=(6,4))
    #plt.plot(x_norm, ratio, label='Matrix / Flagellated', color='black')
    #plt.xlabel("$r_{norm}$")
    #plt.ylabel("Ratio Matrix / Flagellated")
    #plt.title(f"{base} - Matrix / Flagellated")
    #plt.legend()
    #plt.tight_layout()
    #plt.savefig(os.path.join(output_dir, f"{base}_Ratio_Normalized.pdf"))
    #plt.close()
    
    # Plot ratio YFP / mCherry
    plt.figure(figsize=(9,6))
    plt.plot(x_norm, ratio_2, label='FM-ratio', color='black')
    plt.xlabel("$r_{norm}$")
    plt.ylabel("$FM-ratio$")
    #plt.title(f"{base} - Flagellated / Matrix")
    #plt.legend()
    plt.xlim(0, 1)  # set x-axis limit to show full range of $r_{norm}$
    plt.ylim(0, 5)  # set y-axis limit to start at 0 and end slightly above max ratio
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f"{base}_Ratio_Normalized_Inverted.pdf"))
    plt.close()
    
    # Plot both ratios in the same graph
    plt.figure(figsize=(9,6))
    plt.plot(x_norm, ratio, label='MF-ratio', color='grey')
    plt.plot(x_norm, ratio_2, label='FM-ratio', color='black')
    plt.xlabel("$r_{norm}$")
    plt.ylabel("$FM-ratio$")
    #plt.title(f"{base} - Flagellated / Matrix")
    #plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f"{base}_combined_Ratio_Normalized.pdf"))
    plt.close()

    # Plot normalized YFP
    #plt.figure(figsize=(6,4))
    #plt.plot(x_norm, yfp_normalized, label='Flagellated / Constitutive', color='purple')
    #plt.xlabel("$r_{norm}$")
    #plt.ylabel("Normalised Flagellated (a.u.)")
    #plt.title(f"{base} - Normalised Flagellated")
    #plt.legend()
    #plt.tight_layout()
    #plt.savefig(os.path.join(output_dir, f"{base}_YFP_to_Constitutive_Normalized.pdf"))
    #plt.close()

    print(f"Normalized plots saved for {base} in {output_dir}")

print("All normalized plots saved in their respective subfolders.")



## Plot Alpha, Beta, Phi, and Theta (the extrema of the profiles) as a function of glycerol or glutamate concentration

In [ ]:
import os
import re
import pandas as pd
import matplotlib.pyplot as plt
from tkinter import Tk, filedialog, simpledialog
import numpy as np
from sklearn.linear_model import LinearRegression
from scipy.signal import savgol_filter, find_peaks


local_window = 1 #Savitzky-Golay filter window for peak detection
min_prominence = 0.005 #Minimum prominence for peak detection
min_prominence_signal = 0.005 
smooth_window = 7 #Savitzky-Golay filter window for smoothing before peak detection
polyorder = 3 #Polynomial order for Savitzky-Golay filter
day =2 #day the image was taken on
slope_threshold = 8 #was 5
slope_threshold_signal = 150 


min_norm = 0 #consider all peaks with a normalized radius above this value
max_norm = 1 #consider all peaks with a normalized radius below this value

plt.rcParams.update({'font.size': 28, 'lines.linewidth': 4}) #define font size and line width for all plots

constant_conc = "1xGlu" #which of the concnetrations is constant in the experiment? Choose either "1xGlu" or "1xGly"

if constant_conc == "1xGly":
    variable_conc = "xGlu"
else:
    variable_conc = "xGly"

output_dir = '/Users/Vincent/Documents/Masterarbeit_Cambridge/Images/Coloured_Images/Results_Max_Ratio_and_Position' #output directory

# Create output directory if it doesn't exist
def select_folders():
    root = Tk()
    root.withdraw()
    root_dir = filedialog.askdirectory(title="Select root folder containing subfolders")
    if not root_dir:
        raise SystemExit("No folder selected.")
    subfolders = [f.path for f in os.scandir(root_dir) if f.is_dir()]
    print("Found subfolders:")
    for i, f in enumerate(subfolders):
        print(f"{i}: {f}")
    selected = simpledialog.askstring(
        "Select Folders",
        "Enter which subfolders should be included (comma separated):"
    )
    if not selected:
        raise SystemExit("No subfolders selected.")
    indices = [int(i.strip()) for i in selected.split(",")]
    return [subfolders[i] for i in indices]


def collect_csv_files(subfolders):
    all_csv = []
    for sf in subfolders:
        for root, dirs, files in os.walk(sf):
            for f in files:
                if f.endswith(".csv"):
                    all_csv.append(os.path.join(root, f))
    return all_csv

# Function to find valid peaks based on slope criteria
def find_valid_peak(values, xvals, norm_vals):
    smooth_values = savgol_filter(values.values, smooth_window, polyorder)
    smooth_series = pd.Series(smooth_values, index=values.index).reset_index(drop=True)
    xvals = xvals.reset_index(drop=True)
    norm_vals = norm_vals.reset_index(drop=True)
    strong_peaks, _ = find_peaks(smooth_series, prominence=min_prominence)
    all_peaks, _ = find_peaks(smooth_series)
    peak_positions = np.unique(np.concatenate((strong_peaks, all_peaks)))
    if smooth_series.iloc[-1] > smooth_series.iloc[-2]:
        peak_positions = np.append(peak_positions, len(smooth_series)-1)
    if len(peak_positions) == 0:
        return None
    peak_indices = smooth_series.iloc[peak_positions].sort_values(ascending=False).index
    for idx in peak_indices:
        local_idx = smooth_series.index.get_loc(idx)
        left = max(0, local_idx - 3)
        right = min(len(smooth_series), local_idx + 4)
        if right - left < 5:
            continue
        y_window = smooth_series.iloc[left:right].values
        x_window = norm_vals.iloc[left:right].values.reshape(-1,1)
        model = LinearRegression().fit(x_window, y_window)
        slope = model.coef_[0]
        if -slope_threshold <= slope <= slope_threshold:
            fitted_y = model.predict(x_window)
            real_norm = norm_vals.iloc[left:right].values
            return smooth_series.loc[idx], xvals.loc[idx], norm_vals.loc[idx], real_norm, fitted_y
    return None




# Function to find valid peaks based on slope criteria for intensity values
def find_valid_peak_intensity(values, xvals, norm_vals):
    smooth_values = savgol_filter(values.values, smooth_window, polyorder)
    smooth_series = pd.Series(smooth_values, index=values.index)  # DO NOT reset_index here
    xvals = xvals.reset_index(drop=True)
    norm_vals = norm_vals.reset_index(drop=True)
    
    strong_peaks, _ = find_peaks(smooth_series, prominence=min_prominence_signal)
    all_peaks, _ = find_peaks(smooth_series)
    peak_positions = np.unique(np.concatenate((strong_peaks, all_peaks)))
    
    if smooth_series.iloc[-1] > smooth_series.iloc[-2]:
        peak_positions = np.append(peak_positions, len(smooth_series)-1)
    
    if len(peak_positions) == 0:
        return None
    
    peak_indices = smooth_series.iloc[peak_positions].sort_values(ascending=False).index
    for idx in peak_indices:
        left = max(0, idx - 3)
        right = min(len(smooth_series), idx + 4)
        if right - left < 5:
            continue

        y_window = smooth_series.iloc[left:right].values
        x_window = norm_vals.iloc[left:right].values.reshape(-1,1)
        model = LinearRegression().fit(x_window, y_window)
        slope = model.coef_[0]
        if -slope_threshold_signal <= slope <= slope_threshold_signal:
            fitted_y = model.predict(x_window)
            # **Return the original $r_{norm}$ of the peak**
            norm_peak = norm_vals.iloc[idx]
            

            return values.iloc[idx], xvals.iloc[idx], norm_peak, norm_vals.iloc[left:right].values, fitted_y
    return None

# Main processing function
def process_files(all_files):
    mcherry_files = [f for f in all_files if f.endswith("_RadialProfile_Angle_mCherry.csv") and constant_conc in f]
    yfp_files = [f for f in all_files if f.endswith("_RadialProfile_Angle_YFP.csv") and constant_conc in f]
    threshold_files = [f for f in all_files if f.endswith("_Threshold.csv") and constant_conc in f]
    results = []

    for mcherry_file in mcherry_files:
        base = os.path.basename(mcherry_file).replace("_RadialProfile_Angle_mCherry.csv", "")
        yfp_file = next((f for f in yfp_files if os.path.basename(f) == f"{base}_RadialProfile_Angle_YFP.csv"), None)
        threshold_file = next((f for f in threshold_files if os.path.basename(f).startswith(base)), None)
        if not yfp_file or not threshold_file:
            continue

        mcherry_df = pd.read_csv(mcherry_file).dropna(how="all").reset_index(drop=True)
        yfp_df = pd.read_csv(yfp_file).dropna(how="all").reset_index(drop=True)
        threshold_df = pd.read_csv(threshold_file).dropna(how="all").reset_index(drop=True)

        if len(mcherry_df) != len(yfp_df):
            print(f"Skipping {base} due to mismatched lengths.")
            continue

        x = mcherry_df["Distance"]
        mcherry_y = mcherry_df["Intensity"]
        yfp_y = yfp_df["Intensity"]
        threshold_y = threshold_df["Intensity"]

        # Determine cutoff index based on threshold crossings
        idx_255 = threshold_y[threshold_y == 255].index
        if len(idx_255) == 0:
            cutoff_index = len(threshold_y) - 1
        else:
            last_255_index = idx_255[-1]
            drops_after_255 = threshold_y[(threshold_y < 15) & (threshold_y.index > last_255_index)].index
            cutoff_index = drops_after_255[0] if len(drops_after_255) > 0 else len(threshold_y) - 1
        cutoff_index = max(0, min(cutoff_index, len(x) - 1))
        cutoff_x = x.iloc[cutoff_index]

        x_tr = x[x <= cutoff_x]
        m_tr = mcherry_y.iloc[:len(x_tr)]
        y_tr = yfp_y.iloc[:len(x_tr)]

        edge_distance = x_tr.iloc[-1]
        if edge_distance == 0:
            continue
        
        # Calculate ratios and handle division by zero safely
        ratio = pd.to_numeric(m_tr / y_tr.replace(0, pd.NA), errors="coerce").fillna(0)
        ratio2 = pd.to_numeric(y_tr / m_tr.replace(0, pd.NA), errors="coerce").fillna(0)

        norm_radii = x_tr / edge_distance
        valid_indices = (norm_radii >= 0.1) & (norm_radii <= 0.98)
        if not valid_indices.any():
            continue

        ratio_valid = ratio[valid_indices]
        ratio2_valid = ratio2[valid_indices]
        x_valid = x_tr[valid_indices]
        norm_radii_valid = norm_radii[valid_indices]

        #find ratio peaks
        peak1 = find_valid_peak(ratio_valid, x_valid, norm_radii_valid)
        peak2 = find_valid_peak(ratio2_valid, x_valid, norm_radii_valid)
        

        #find intensity peaks 
        peak1_intensity = find_valid_peak_intensity(m_tr, x_tr, norm_radii)
        peak2_intensity = find_valid_peak_intensity(y_tr, x_tr, norm_radii)

        if peak1 is None or peak2 is None or peak1_intensity is None or peak2_intensity is None:
            continue

        max_ratio, max_radius, norm_max_radius, fit_norm1, fit_y1 = peak1
        max_ratio2, max_radius2, norm_max_radius2, fit_norm2, fit_y2 = peak2
        max_mcherry_intensity, max_mcherry_radius, max_mcherry_norm, fit_norm_mcherry, fit_y_mcherry = peak1_intensity
        max_yfp_intensity, max_yfp_radius, max_yfp_norm, fit_norm_yfp, fit_y_yfp = peak2_intensity

    




        curve_obj = {
            "norm_r": norm_radii.values,
            "ratio": ratio.values,
            "fit_norm": fit_norm1,
            "fit_y": fit_y1
        }

        Y_signal_obj = {
            "norm_r": norm_radii.values,
            "intensity": y_tr.values,
        }
        M_cherry_obj = {
            "norm_r": norm_radii.values,
            "intensity": m_tr.values,
        }



        if constant_conc == "1xGly":
            match = re.search(r"1xGly_(\d*\.?\d*)xGlu", base)
        else:
            match = re.search(r"(\d*\.?\d*)xGly_1xGlu", base)
        if not match:
            continue
        concentration = float(match.group(1))
        if concentration == 0:
            continue

        results.append({
            "Base": base,
            "Concentration": concentration,
            "MinRatio_Flagellated_Matrix": 1/max_ratio,
            "MaxRatio_Flagellated_Matrix": max_ratio2,
            "MaxMatrixIntensity": max_mcherry_intensity,
            "MaxFlagellatedIntensity": max_yfp_intensity,
            "EdgeDistance": edge_distance,
            "NormMaxRadius": norm_max_radius,
            "NormMaxRadius2": norm_max_radius2,
            "NormMCherryMaxRadius": max_mcherry_norm,
            "NormYFPMaxRadius": max_yfp_norm,
            "CurveData": curve_obj,
            "YFP_Data": Y_signal_obj,
            "MCherry_Data": M_cherry_obj
        })

    return pd.DataFrame(results) if results else None


def plot_results(df):
    grouped = df.groupby("Concentration").agg(
        MeanMinRatio2=("MinRatio_Flagellated_Matrix", "mean"),
        SDMinRatio2=("MinRatio_Flagellated_Matrix", "std"),
        MeanMaxRatio2=("MaxRatio_Flagellated_Matrix", "mean"),
        SDMaxRatio2=("MaxRatio_Flagellated_Matrix", "std"),
        MeanEdgeDistance=("EdgeDistance", "mean"),
        MeanNormRadius=("NormMaxRadius", "mean"),
        SDNormRadius=("NormMaxRadius", "std"),
        MeanNormRadius2=("NormMaxRadius2", "mean"),
        SDNormRadius2=("NormMaxRadius2", "std")
    ).reset_index()

    intensity_grouped = df.groupby("Concentration").agg(
        MeanMatrix=("MaxMatrixIntensity", "mean"),
        SDMatrix=("MaxMatrixIntensity", "std"),
        MeanFlagellated=("MaxFlagellatedIntensity", "mean"),
        SDFlagellated=("MaxFlagellatedIntensity", "std")
    ).reset_index()

    plt.figure(figsize=(10, 8))
    plt.errorbar(intensity_grouped["Concentration"], intensity_grouped["MeanMatrix"],
                 yerr=intensity_grouped["SDMatrix"], fmt='o', capsize=5, color='red',
                 label='$I_{m,\\beta}$')
    plt.errorbar(intensity_grouped["Concentration"], intensity_grouped["MeanFlagellated"],
                 yerr=intensity_grouped["SDFlagellated"], fmt='o', capsize=5, color='orange',
                 label='$I_{f,\\alpha}$')
    for conc, subdf in df.groupby("Concentration"):
        plt.plot([conc]*len(subdf), subdf["MaxMatrixIntensity"], 'x', color='red')
        plt.plot([conc]*len(subdf), subdf["MaxFlagellatedIntensity"], 'x', color='orange')
    plt.xlabel("Concentration " + "(" + variable_conc + ")")
    plt.ylabel("Intensity (a.u.)")
    if day == 5:
        plt.ylim(0, 90)
    else:
        plt.ylim(0, 160)
    plt.xlim(-0.05,1.6)
    if day == 5:
        if constant_conc == "1xGly":
            plt.vlines(0.2, 0, 90, color='grey', linestyle='--',linewidths=3)
            plt.vlines(0.5, 0, 90, color='grey', linestyle='--', linewidths=3)
    #plt.title(f"Max Fluorescence Intensity vs Concentration Day {day}")
    plt.legend(fontsize='small')
    plt.savefig(os.path.join(output_dir, f"Max_Intensity_vs_concentration_day_{day}_{constant_conc}.pdf"))
    plt.show()




    plt.figure(figsize=(10, 8))
    plt.errorbar(intensity_grouped["Concentration"], intensity_grouped["MeanMatrix"],
                 yerr=intensity_grouped["SDMatrix"], fmt='o', capsize=5, color='red',
                 label='Max Matrix Intensity')
    for conc, subdf in df.groupby("Concentration"):
        plt.plot([conc]*len(subdf), subdf["MaxMatrixIntensity"], 'x', color='red')
    plt.xlabel("Concentration " + "(" + variable_conc + ")")
    plt.ylabel("Intensity (a.u.)")
    plt.ylim(0, (intensity_grouped["MeanMatrix"].max() + intensity_grouped["SDMatrix"].max()) * 1.1)
    plt.xlim(-0.05,1.6)
    #plt.title(f"Max Fluorescence Intensity vs Concentration Day {day}")
    plt.legend(fontsize='small')
    plt.savefig(os.path.join(output_dir, f"Max_Matrix_Intensity_vs_concentration_day_{day}_{constant_conc}.pdf"))
    plt.show()




    plt.figure(figsize=(10, 8))
    plt.errorbar(intensity_grouped["Concentration"], intensity_grouped["MeanFlagellated"],
                 yerr=intensity_grouped["SDFlagellated"], fmt='o', capsize=5, color='orange',
                 label='Max Flagellated Intensity')
    for conc, subdf in df.groupby("Concentration"):
        plt.plot([conc]*len(subdf), subdf["MaxFlagellatedIntensity"], 'x', color='orange')
    plt.xlabel("Concentration " + "(" + variable_conc + ")")
    plt.ylabel("Intensity (a.u.)")
    plt.ylim(0, (intensity_grouped["MeanFlagellated"].max() + intensity_grouped["SDFlagellated"].max()) * 1.1)
    plt.xlim(-0.05,1.6)
    #plt.title(f"Max Fluorescence Intensity vs Concentration Day {day}")
    plt.legend(fontsize='small')
    plt.savefig(os.path.join(output_dir, f"Max_FlagellatedIntensity_vs_concentration_day_{day}_{constant_conc}.pdf"))
    plt.show()



    plt.figure(figsize=(10, 8))
    plt.errorbar(grouped["Concentration"], grouped["MeanMinRatio2"],
                 yerr=grouped["SDMinRatio2"], fmt='o', capsize=5, color='black')
    for conc, subdf in df.groupby("Concentration"):
        plt.plot([conc]*len(subdf), subdf["MinRatio_Flagellated_Matrix"], 'x', color='black')
    plt.xlabel("Concentration " + "(" + variable_conc + ")")
    plt.ylabel("Min FM-ratio")
    plt.title(f"Min FM-ratio vs Concentration Day {day}")
    plt.ylim(0, 2)
    plt.xlim(-0.05,1.6)
    plt.savefig(os.path.join(output_dir, f"Min_FM_Ratio_vs_concentration_day_{day}_{constant_conc}.pdf"))
    plt.show()



    plt.figure(figsize=(10, 8))
    plt.errorbar(grouped["Concentration"], grouped["MeanMaxRatio2"],
                 yerr=grouped["SDMaxRatio2"], fmt='o', capsize=5, color='slategrey')
    for conc, subdf in df.groupby("Concentration"):
        plt.plot([conc]*len(subdf), subdf["MaxRatio_Flagellated_Matrix"], 'x', color='slategrey')
    plt.xlabel("Concentration " + "(" + variable_conc + ")")
    plt.ylabel("Max FM-ratio")
    if day == 5:
        plt.ylim(0, 6)
    else:   
        plt.ylim(0, 4) 
    plt.xlim(-0.05,1.6)
    #plt.title(f"Max FM-ratio vs Concentration Day {day}")
    plt.savefig(os.path.join(output_dir, f"Max_FM_Ratio_vs_concentration_day_{day}_{constant_conc}.pdf"))
    plt.show()



    plt.figure(figsize=(10, 8))
    plt.errorbar(grouped["Concentration"], grouped["MeanMinRatio2"], yerr=grouped["SDMinRatio2"],
                 fmt='o', capsize=5, color='black', label='$I_{FM,\\theta}$')
    plt.errorbar(grouped["Concentration"], grouped["MeanMaxRatio2"], yerr=grouped["SDMaxRatio2"],
                 fmt='o', capsize=5, color='slategrey', label='$I_{FM,\\phi}$')
    for conc, subdf in df.groupby("Concentration"):
        plt.plot([conc]*len(subdf), subdf["MinRatio_Flagellated_Matrix"], 'x', color='black')
        plt.plot([conc]*len(subdf), subdf["MaxRatio_Flagellated_Matrix"], 'x', color='slategrey')
    plt.xlabel("Concentration " + "(" + variable_conc + ")")
    plt.ylabel("FM-ratio")
    if day == 5:
        plt.ylim(0, 6)
        if constant_conc == "1xGly":
            plt.vlines(0.2, 0, 6, color='grey', linestyle='--', linewidths=3)
            plt.vlines(0.5, 0, 6, color='grey', linestyle='--', linewidths=3)
    else:   
        plt.ylim(0, 4) 
    plt.xlim(-0.05,1.6)
    #plt.title(f"Max FM-ratio vs Concentration Day {day}")
    plt.legend(fontsize='small')
    plt.savefig(os.path.join(output_dir, f"Max_Ratio_vs_concentration_day_{day}_{constant_conc}.pdf"))
    plt.show()



    plt.figure(figsize=(10, 8))
    plt.errorbar(grouped["Concentration"], grouped["MeanNormRadius"],
                 yerr=grouped["SDNormRadius"], fmt='o', capsize=5, color='black')
    for conc, subdf in df.groupby("Concentration"):
        plt.plot([conc]*len(subdf), subdf["NormMaxRadius"], 'x', color='black')
    plt.ylim(0,1)
    plt.xlim(-0.05,1.6)
    plt.xlabel("Concentration " + "(" + variable_conc + ")")
    plt.ylabel("$r_{norm}$")
    #plt.title(f"$r_{norm}$ Matrix/Flagellated vs Concentration Day {day}")
    plt.savefig(os.path.join(output_dir, f"Position_Max_MF_vs_concentration_day_{day}_{constant_conc}.pdf"))
    plt.show()



    plt.figure(figsize=(10, 8))
    plt.errorbar(grouped["Concentration"], grouped["MeanNormRadius2"],
                 yerr=grouped["SDNormRadius2"], fmt='o', capsize=5, color='slategrey')
    for conc, subdf in df.groupby("Concentration"):
        plt.plot([conc]*len(subdf), subdf["NormMaxRadius2"], 'x', color='slategrey')
    plt.ylim(0,1)
    plt.xlim(-0.05,1.6)
    plt.xlabel("Concentration " + "(" + variable_conc + ")")
    plt.ylabel("$r_{norm}$")
    #plt.title(f"$r_{norm}$ Flagellated/Matrix vs Concentration Day {day}")
    plt.savefig(os.path.join(output_dir, f"Position_Max_FM_vs_concentration_day_{day}_{constant_conc}.pdf"))
    plt.show()



    plt.figure(figsize=(10, 8))
    plt.errorbar(grouped["Concentration"], grouped["MeanNormRadius"], yerr=grouped["SDNormRadius"],
                 fmt='o', capsize=5, color='black', label='$r_{\\theta}$')
    plt.errorbar(grouped["Concentration"], grouped["MeanNormRadius2"], yerr=grouped["SDNormRadius2"],
                 fmt='o', capsize=5, color='slategrey', label='$r_{\\phi}$')
    for conc, subdf in df.groupby("Concentration"):
        plt.plot([conc]*len(subdf), subdf["NormMaxRadius"], 'x', color='black')
        plt.plot([conc]*len(subdf), subdf["NormMaxRadius2"], 'x', color='slategrey')
    plt.ylim(0,1)
    plt.xlim(-0.05,1.6)
    plt.xlabel("Concentration " + "(" + variable_conc + ")")
    plt.ylabel("$r_{norm}$")
    if day == 5:
        if constant_conc == "1xGly":
            plt.vlines(0.2, 0, 6, color='grey', linestyle='--',linewidths=3)
            plt.vlines(0.5, 0, 6, color='grey', linestyle='--', linewidths=3)
    #plt.title(f"$r_{norm}$ Matrix/Flagellated vs Concentration Day {day}")
    plt.legend( fontsize='small')
    plt.savefig(os.path.join(output_dir, f"Position_Max_Ratio_vs_concentration_day_{day}_{constant_conc}.pdf"))
    plt.show()

    # Plot individual curves for each sample to check if the peak finding works correctly
    """
    for conc, subdf in df.groupby("Concentration"):
        for _, row in subdf.iterrows():
            curve = row["CurveData"]
            norm_r = curve["norm_r"]
            ratio_curve = curve["ratio"]
            fit_norm = curve["fit_norm"]
            fit_y = curve["fit_y"]
            yfp_signal = row["YFP_Data"]
            mcherry_signal = row["MCherry_Data"]

            # Prepare second ratio (Flagellated/Matrix)
            ratio_curve_2 = np.divide(1, ratio_curve, where=ratio_curve != 0)
            ratio_curve_2[ratio_curve == 0] = np.nan

            mask = (norm_r >= 0) & (norm_r <= 1)

            # Plot both ratios in one figure 
            print(f"Ratios – {row['Base']} ({conc}{variable_conc})")
            plt.figure(figsize=(9, 6))
            #plt.plot(norm_r[mask], ratio_curve[mask], '-', color='blue', label="Matrix/Flagellated")
            #plt.plot(fit_norm, fit_y, '-', color='red', linewidth=3, label="Fitted Region MF")
            plt.scatter(row["NormMaxRadius"], row["MinRatio_Flagellated_Matrix"],
                        color='red', marker='v', linewidths=6, zorder=10, label="$(r_{\\theta},I_{FM,\\theta}$)")

            # Fit for Flagellated/Matrix ratio
            center = np.argmin(np.abs(norm_r - row["NormMaxRadius2"]))
            if center >= 3 and center < len(norm_r) - 3:
                y2_window = ratio_curve_2[center-3:center+4]
                x2_window = norm_r[center-3:center+4]
                valid = np.isfinite(y2_window)
                if np.sum(valid) >= 3:
                    model2 = LinearRegression().fit(x2_window[valid].reshape(-1,1), y2_window[valid])
                    fit_y2 = model2.predict(x2_window.reshape(-1,1))
                    #plt.plot(x2_window, fit_y2, '-', color='orange', linewidth=3, label="Fitted Region FM")
            plt.plot(norm_r[mask], ratio_curve_2[mask], '-', color='green', label="$I_{FM}(r_{norm})$")
            plt.scatter(row["NormMaxRadius2"], row["MaxRatio_Flagellated_Matrix"],
                        color='orange', marker='^', linewidths=6, zorder=10, label="$(r_{\\phi},I_{FM,\\phi}$)")

            plt.xlabel("$r_{norm}$")
            plt.ylabel("$FM$-$ratio$")
            plt.xlim(0, 1)  # set x-axis limit to show full range of $r_{norm}$
            plt.ylim(0,5)
            #plt.title(f"Ratios – {row['Base']} ({conc}{variable_conc})")
            plt.legend(fontsize='small')
            plt.show()

            # Get normalized radii for the intensity signals
            norm_r_m = row["MCherry_Data"]["norm_r"]
            norm_r_y = row["YFP_Data"]["norm_r"]

            m_tr = row["MCherry_Data"]["intensity"]
            y_tr = row["YFP_Data"]["intensity"]

            plt.figure(figsize=(9, 6))
            plt.plot(norm_r_m, m_tr, '-', color='red', label="$I_m(r_{norm})$")
            plt.plot(norm_r_y, y_tr, '-', color='orange', label="$I_f(r_{norm})$")
            plt.scatter(row["NormMCherryMaxRadius"], row["MaxMatrixIntensity"], color='black', zorder=10, linewidths=6, label="$(r_{\\beta},I_{m,\\beta}$)")
            plt.scatter(row["NormYFPMaxRadius"], row["MaxFlagellatedIntensity"], color='grey', zorder=10, linewidths=6, label="$(r_{\\alpha},I_{f,\\alpha}$)")

            plt.xlabel("$r_{norm}$")
            plt.ylabel("$Intensity \ (a.u.)$")
            plt.xlim(0, 1)  # set x-axis limit to show full range of $r_{norm}$
            plt.ylim(0,75)
            #plt.title(f"Intensities – {row['Base']} ({conc}{variable_conc})")
            plt.legend(fontsize='small')
            plt.show() 
    """

selected_subfolders = select_folders()
all_files = collect_csv_files(selected_subfolders)
df = process_files(all_files)

if df is not None and not df.empty:
    plot_results(df)
else:
    print("No results found.")

# Calculate the integrals of the fluorecent reporter profiles (A), and the ratio of the integrals (psi)


In [ ]:
import os
import re
import pandas as pd
import matplotlib.pyplot as plt
from tkinter import Tk, filedialog, simpledialog
import numpy as np
from sklearn.linear_model import LinearRegression
from scipy.signal import savgol_filter, find_peaks



constant_conc = "1xGlu" #which of the concnetrations is constant in the experiment? Choose either "1xGlu" or "1xGly"
day = 2 #day the image was taken on, used for plot titles



local_window = 1 #Savitzky-Golay filter window for peak detection
min_prominence = 0.005 #Minimum prominence for peak detection in the ratio curves 
min_prominence_signal = 0.005 #Minimum prominence for peak detection in the intensity curves
smooth_window = 7 #Savitzky-Golay filter window for smoothing before peak detection
polyorder = 3 #Polynomial order for Savitzky-Golay filter
y_low_lim = 0.5 #lower limit of the y axis
if day == 5:    
    y_high_lim = 2.75 #upper limit of the y axis (was 2.75, 1.5)
else:
    y_high_lim = 1.5 #upper limit of the y axis (was 2.75, 1.5)
slope_threshold = 5 #maximum allowed slope for a peak to be considered valid in the ratio curves, was 5
slope_threshold_signal = 150

min_norm = 0 #consider all peaks with a normalized radius above this value, was 0.1
max_norm = 1 #consider all peaks with a normalized radius below this value



if constant_conc == "1xGly":
    variable_conc = "xGlu"
else:
    variable_conc = "xGly"

output_dir = '/Users/Vincent/Documents/Masterarbeit_Cambridge/Images/Coloured_Images/Results_Max_Ratio_and_Position' #output directory, change this to your desired output path

# Create output directory if it doesn't exist
def select_folders():
    root = Tk()
    root.withdraw()
    root_dir = filedialog.askdirectory(title="Select root folder containing subfolders")
    if not root_dir:
        raise SystemExit("No folder selected.")
    subfolders = [f.path for f in os.scandir(root_dir) if f.is_dir()]
    print("Found subfolders:")
    for i, f in enumerate(subfolders):
        print(f"{i}: {f}")
    selected = simpledialog.askstring(
        "Select Folders",
        "Enter which subfolders should be included (comma separated):"
    )
    if not selected:
        raise SystemExit("No subfolders selected.")
    indices = [int(i.strip()) for i in selected.split(",")]
    return [subfolders[i] for i in indices]


def collect_csv_files(subfolders):
    all_csv = []
    for sf in subfolders:
        for root, dirs, files in os.walk(sf):
            for f in files:
                if f.endswith(".csv"):
                    all_csv.append(os.path.join(root, f))
    return all_csv

# Function to find valid peaks based on slope criteria
def find_valid_peak(values, xvals, norm_vals):

    smooth_values = savgol_filter(values.values, smooth_window, polyorder)
    smooth_series = pd.Series(smooth_values, index=values.index).reset_index(drop=True)
    xvals = xvals.reset_index(drop=True)
    norm_vals = norm_vals.reset_index(drop=True)

    strong_peaks, _ = find_peaks(smooth_series, prominence=min_prominence)
    all_peaks, _ = find_peaks(smooth_series)

    peak_positions = np.unique(np.concatenate((strong_peaks, all_peaks)))

    if smooth_series.iloc[-1] > smooth_series.iloc[-2]:
        peak_positions = np.append(peak_positions, len(smooth_series)-1)

    if len(peak_positions) == 0:
        return None

    peak_indices = smooth_series.iloc[peak_positions].sort_values(ascending=False).index

    for idx in peak_indices:

        local_idx = smooth_series.index.get_loc(idx)

        left = max(0, local_idx - 3)
        right = min(len(smooth_series), local_idx + 4)

        if right - left < 5:
            continue

        y_window = smooth_series.iloc[left:right].values
        x_window = norm_vals.iloc[left:right].values.reshape(-1,1)

        model = LinearRegression().fit(x_window, y_window)
        slope = model.coef_[0]

        if -slope_threshold <= slope <= slope_threshold:

            fitted_y = model.predict(x_window)
            real_norm = norm_vals.iloc[left:right].values

            return smooth_series.loc[idx], xvals.loc[idx], norm_vals.loc[idx], real_norm, fitted_y

    return None

# Function to find valid peaks based on slope criteria for intensity values
def find_valid_peak_intensity(values, xvals, norm_vals):

    smooth_values = savgol_filter(values.values, smooth_window, polyorder)
    smooth_series = pd.Series(smooth_values, index=values.index)

    xvals = xvals.reset_index(drop=True)
    norm_vals = norm_vals.reset_index(drop=True)

    strong_peaks, _ = find_peaks(smooth_series, prominence=min_prominence_signal)
    all_peaks, _ = find_peaks(smooth_series)

    peak_positions = np.unique(np.concatenate((strong_peaks, all_peaks)))

    if smooth_series.iloc[-1] > smooth_series.iloc[-2]:
        peak_positions = np.append(peak_positions, len(smooth_series)-1)

    if len(peak_positions) == 0:
        return None

    peak_indices = smooth_series.iloc[peak_positions].sort_values(ascending=False).index

    for idx in peak_indices:

        left = max(0, idx - 3)
        right = min(len(smooth_series), idx + 4)

        if right - left < 5:
            continue

        y_window = smooth_series.iloc[left:right].values
        x_window = norm_vals.iloc[left:right].values.reshape(-1,1)

        model = LinearRegression().fit(x_window, y_window)
        slope = model.coef_[0]

        if -slope_threshold_signal <= slope <= slope_threshold_signal:

            fitted_y = model.predict(x_window)

            norm_peak = norm_vals.iloc[idx]

            return values.iloc[idx], xvals.iloc[idx], norm_peak, norm_vals.iloc[left:right].values, fitted_y

    return None


def process_files(all_files):

    mcherry_files = [f for f in all_files if f.endswith("_RadialProfile_Angle_mCherry.csv") and constant_conc in f]
    yfp_files = [f for f in all_files if f.endswith("_RadialProfile_Angle_YFP.csv") and constant_conc in f]
    threshold_files = [f for f in all_files if f.endswith("_Threshold.csv") and constant_conc in f]

    results = []

    for mcherry_file in mcherry_files:

        base = os.path.basename(mcherry_file).replace("_RadialProfile_Angle_mCherry.csv", "")

        yfp_file = next((f for f in yfp_files if os.path.basename(f) == f"{base}_RadialProfile_Angle_YFP.csv"), None)
        threshold_file = next((f for f in threshold_files if os.path.basename(f).startswith(base)), None)

        if not yfp_file or not threshold_file:
            continue

        mcherry_df = pd.read_csv(mcherry_file).dropna(how="all").reset_index(drop=True)
        yfp_df = pd.read_csv(yfp_file).dropna(how="all").reset_index(drop=True)
        threshold_df = pd.read_csv(threshold_file).dropna(how="all").reset_index(drop=True)

        if len(mcherry_df) != len(yfp_df):
            continue

        x = mcherry_df["Distance"]
        mcherry_y = mcherry_df["Intensity"]
        yfp_y = yfp_df["Intensity"]
        threshold_y = threshold_df["Intensity"]

        # Determine cutoff index based on threshold crossings
        idx_255 = threshold_y[threshold_y == 255].index

        if len(idx_255) == 0:
            cutoff_index = len(threshold_y) - 1
        else:
            last_255_index = idx_255[-1]
            drops_after_255 = threshold_y[(threshold_y < 15) & (threshold_y.index > last_255_index)].index #find indices where threshold drops below 15 after last 255
            cutoff_index = drops_after_255[0] if len(drops_after_255) > 0 else len(threshold_y) - 1

        cutoff_x = x.iloc[cutoff_index]

        x_tr = x[x <= cutoff_x]
        m_tr = mcherry_y.iloc[:len(x_tr)]
        y_tr = yfp_y.iloc[:len(x_tr)]

        edge_distance = x_tr.iloc[-1]

        if edge_distance == 0:
            continue

        norm_radii = x_tr / edge_distance


        # calculate the integrals using a riemann sum approach
        
        r = norm_radii.values
        r_2 = x_tr.values #actual radius
        I_matrix = m_tr.values
        I_yfp = y_tr.values

        dr = np.gradient(r)
        dr_2 = np.gradient(r_2)

        matrix_integral = np.sum(I_matrix * r_2 * dr) / (np.pi * cutoff_x * cutoff_x) #normalize by the area of the circle with radius equal to the edge distance
        yfp_integral = np.sum(I_yfp * r_2 * dr) / (np.pi * cutoff_x * cutoff_x) #normalize by the area of the circle with radius equal to the edge distance
        #matrix_integral = np.trapz(m_tr*norm_radii, norm_radii)
        #yfp_integral = np.trapz(y_tr*norm_radii, norm_radii)

        if matrix_integral == 0:
            integral_ratio = np.nan
        else:
            integral_ratio = yfp_integral / matrix_integral


        ratio = pd.to_numeric(m_tr / y_tr.replace(0, pd.NA) , errors="coerce").fillna(0)
        ratio2 = pd.to_numeric(y_tr / m_tr.replace(0, pd.NA), errors="coerce").fillna(0)

        valid_indices = (norm_radii >= 0.15) & (norm_radii <= 0.99)

        if not valid_indices.any():
            continue

        ratio_valid = ratio[valid_indices]
        ratio2_valid = ratio2[valid_indices]

        x_valid = x_tr[valid_indices]
        norm_radii_valid = norm_radii[valid_indices]

        peak1 = find_valid_peak(ratio_valid, x_valid, norm_radii_valid)
        peak2 = find_valid_peak(ratio2_valid, x_valid, norm_radii_valid)

        peak1_intensity = find_valid_peak_intensity(m_tr, x_tr, norm_radii)
        peak2_intensity = find_valid_peak_intensity(y_tr, x_tr, norm_radii)

        if peak1 is None or peak2 is None or peak1_intensity is None or peak2_intensity is None:
            continue

        max_ratio, max_radius, norm_max_radius, fit_norm1, fit_y1 = peak1
        max_ratio2, max_radius2, norm_max_radius2, fit_norm2, fit_y2 = peak2

        max_mcherry_intensity, max_mcherry_radius, max_mcherry_norm, fit_norm_mcherry, fit_y_mcherry = peak1_intensity
        max_yfp_intensity, max_yfp_radius, max_yfp_norm, fit_norm_yfp, fit_y_yfp = peak2_intensity

        if constant_conc == "1xGly":
            match = re.search(r"1xGly_(\d*\.?\d*)xGlu", base)
        else:
            match = re.search(r"(\d*\.?\d*)xGly_1xGlu", base)

        if not match:
            continue

        concentration = float(match.group(1))

        if concentration == 0:
            continue

        results.append({

            "Base": base,
            "Concentration": concentration,

            "MinRatio_Flagellated_Matrix": max_ratio,
            "MaxRatio_Flagellated_Matrix": max_ratio2,

            "MaxMatrixIntensity": max_mcherry_intensity,
            "MaxFlagellatedIntensity": max_yfp_intensity,

            #
            "MatrixIntegral": matrix_integral,
            "YFPIntegral": yfp_integral,
            "IntegralRatio_YFP_Matrix": integral_ratio,

            "EdgeDistance": edge_distance,

            "NormMaxRadius": norm_max_radius,
            "NormMaxRadius2": norm_max_radius2,

            "NormMCherryMaxRadius": max_mcherry_norm,
            "NormYFPMaxRadius": max_yfp_norm

        })

    return pd.DataFrame(results) if results else None


def plot_results(df):

    integral_grouped = df.groupby("Concentration").agg(
        MeanMatrixIntegral=("MatrixIntegral", "mean"),
        SDMatrixIntegral=("MatrixIntegral", "std"),
        MeanYFPIntegral=("YFPIntegral", "mean"),
        SDYFPIntegral=("YFPIntegral", "std"),
        MeanIntegralRatio=("IntegralRatio_YFP_Matrix", "mean"),
        SDIntegralRatio=("IntegralRatio_YFP_Matrix", "std")
    ).reset_index()


    # plot the integrals 

    plt.figure(figsize=(10,8))

    plt.errorbar(integral_grouped["Concentration"],
                 integral_grouped["MeanMatrixIntegral"],
                 yerr=integral_grouped["SDMatrixIntegral"],
                 fmt='o', capsize=5, color='red', label='$O_m$')

    plt.errorbar(integral_grouped["Concentration"],
                 integral_grouped["MeanYFPIntegral"],
                 yerr=integral_grouped["SDYFPIntegral"],
                 fmt='o', capsize=5, color='orange', label='$O_f$')

    for conc, subdf in df.groupby("Concentration"):
        plt.plot([conc]*len(subdf), subdf["MatrixIntegral"], 'x', color='red')
        plt.plot([conc]*len(subdf), subdf["YFPIntegral"], 'x', color='orange')

    plt.xlabel("Concentration (" + variable_conc + ")")
    plt.ylabel("O")
    if day == 5:    
        plt.ylim(0, 0.025) #0.05 for day 2, 0.025 for day 5
    else:
        plt.ylim(0, 0.05)
    plt.xlim(-0.05, 1.6)
    if day == 5:
        if constant_conc == "1xGly":
            plt.vlines(0.2, 0, 6, color='grey', linestyle='--', linewidths=3)
            plt.vlines(0.5, 0, 6, color='grey', linestyle='--', linewidths=3)

    plt.legend()

    plt.savefig(os.path.join(output_dir,
        f"Integral_Intensity_vs_concentration_day_{day}_{constant_conc}.pdf"))

    plt.show()


    # plot the ratio of integrasl


    plt.figure(figsize=(10,8))

    plt.errorbar(integral_grouped["Concentration"],
                 integral_grouped["MeanIntegralRatio"],
                 yerr=integral_grouped["SDIntegralRatio"],
                 fmt='o', capsize=5, color='black')

    for conc, subdf in df.groupby("Concentration"):
        plt.plot([conc]*len(subdf),
                 subdf["IntegralRatio_YFP_Matrix"],
                 'x', color='black')

    plt.xlabel("Concentration (" + variable_conc + ")")
    plt.ylabel("$\\psi$")

    plt.ylim(y_low_lim, y_high_lim)
    plt.xlim(-0.05, 1.6)
    if day == 5:
        if constant_conc == "1xGly":
            plt.vlines(0.2, y_low_lim, y_high_lim, color='grey', linestyle='--', linewidths=3)
            plt.vlines(0.5, y_low_lim, y_high_lim, color='grey', linestyle='--', linewidths=3)

    plt.savefig(os.path.join(output_dir,
        f"Integral_Ratio_vs_concentration_day_{day}_{constant_conc}.pdf"))

    plt.show()


selected_subfolders = select_folders()
all_files = collect_csv_files(selected_subfolders)

df = process_files(all_files)

if df is not None and not df.empty:
    plot_results(df)
else:
    print("No results found.")

# Data Analysis for Timelapse Images

### Select the Folder containing the timelapse data

In [ ]:
import ipywidgets as widgets
from IPython.display import display

path_text = widgets.Textarea(
    value='',
    placeholder='Paste or type folder path here',
    description='Folder:',
    disabled=False,
    layout=widgets.Layout(width='600px')
)

button = widgets.Button(description="Confirm")

def on_button_click(b):
    global input_dir
    input_dir = path_text.value.strip()
    print("Selected folder:", input_dir)

button.on_click(on_button_click)

display(path_text, button)


### Plot the Individual Graphs for every Frame in the Timelapse, including the Ratios. Then, also Plot the Ratios for every 5 frames in the same Graph as well as the YFP Plots


In [ ]:
import os
import re
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import cmcrameri.cm as cmc
from io import BytesIO
import imageio as imageio
plt.rcParams.update({'font.size': 30, 'font.family': 'Mathematica1', 'lines.linewidth': 4})

# Check if input_dir is set and valid
if "input_dir" not in globals() or not os.path.isdir(input_dir):
    raise SystemExit("No valid folder selected. Run the first cell and click Confirm.")


# Helper: parse filenames

pattern = re.compile(
    r"^(?P<base>.+)_(?P<channel>mCherry|YFP|Threshold)_(?P<index>\d+)\.csv$",
    re.IGNORECASE,
)


grouped = {}

# Collect & group files

for root, dirs, files in os.walk(input_dir):
    for f in files:
        match = pattern.match(f)
        if not match:
            continue

        base = match.group("base")
        channel = match.group("channel").capitalize()
        index = int(match.group("index"))
        full_path = os.path.join(root, f)

        grouped.setdefault(base, {})
        grouped[base].setdefault(index, {})
        grouped[base][index][channel] = full_path

# Storage for in-memory GIF frames
gif_mcherry_frames = []
gif_yfp_frames = []
gif_combined_frames = []
gif_combined_ratio_frames = []

# Processing

for base, indices in grouped.items():

    print(f"\nProcessing dataset group: {base}")

    # Prepare output directory
    some_file = next(iter(next(iter(indices.values())).values()))
    subfolder = os.path.dirname(some_file)
    output_dir = os.path.join(subfolder, "Plots", base)
    os.makedirs(output_dir, exist_ok=True)

    # Storage for group plots
    all_mcherry = []
    all_yfp = []
    all_xnorm = []
    all_ratio = []
    all_ratio_inv = []
    valid_indices = []

    # Process each frame

    for index, files_dict in sorted(indices.items()):

        print(f"  Index {index}: ", end="")

        if not all(k in files_dict for k in ("Mcherry", "Yfp", "Threshold")):
            print("missing channel(s), skipping.")
            continue

        print("OK")

        # Files
        mcherry_file = files_dict["Mcherry"]
        yfp_file = files_dict["Yfp"]
        threshold_file = files_dict["Threshold"]

        # Read CSV
        mcherry_df = pd.read_csv(mcherry_file).dropna(how='all').reset_index(drop=True)
        yfp_df = pd.read_csv(yfp_file).dropna(how='all').reset_index(drop=True)
        threshold_df = pd.read_csv(threshold_file).dropna(how='all').reset_index(drop=True)

        # Check lengths
        if not (len(mcherry_df) == len(yfp_df) == len(threshold_df)):
            print(f"    Length mismatch for index {index}, skipping.")
            continue

        x = mcherry_df["Distance"]
        mcherry_y = mcherry_df["Intensity"]
        yfp_y = yfp_df["Intensity"]
        threshold_y = threshold_df["Intensity"]

        # updated edge detection

        idx_255 = threshold_y[threshold_y == 255].index

        if len(idx_255) == 0:
            cutoff_index = len(threshold_y) - 1
        else:
            last_255_index = idx_255[-1]

            drops_after_255 = threshold_y[
                (threshold_y < 15) & (threshold_y.index > last_255_index)
            ].index

            if len(drops_after_255) > 0:
                cutoff_index = drops_after_255[0]
            else:
                cutoff_index = len(threshold_y) - 1

        cutoff_index = max(0, min(cutoff_index, len(x) - 1))
        cutoff_x = x.iloc[cutoff_index]

        # Truncate
        x_trunc = x[x <= cutoff_x]
        mcherry_trunc = mcherry_y.iloc[:len(x_trunc)]
        yfp_trunc = yfp_y.iloc[:len(x_trunc)]

        # Normalise x-axis
        x_norm = x_trunc / x_trunc.iloc[-1]

        # Store for group plots
        all_xnorm.append(x_norm)
        all_mcherry.append(mcherry_trunc)
        all_yfp.append(yfp_trunc)
        valid_indices.append(index)

        # Ratios
        ratio = (mcherry_trunc / yfp_trunc.replace(0, pd.NA)).fillna(0)
        ratio_inv = (yfp_trunc.replace(0, pd.NA) / mcherry_trunc.replace(0, pd.NA)).fillna(0)

        all_ratio.append(ratio)
        all_ratio_inv.append(ratio_inv)

        # Save individual plots
        plt.figure(figsize=(9,5))
        plt.plot(x_norm, mcherry_trunc, color='red')
       #plt.title(f"mCherry - frame {index}")
        plt.xlabel("$r_{norm}$")
        plt.ylabel("$Intensity \ (a. u.)$")
        pdf_path = os.path.join(output_dir, f"{base}_mCherry_{index}_normalized.pdf")
        plt.savefig(pdf_path)

        buf = BytesIO()
        plt.savefig(buf, format='png')
        buf.seek(0)
        gif_mcherry_frames.append(imageio.imread(buf))
        buf.close()
        plt.close()

        plt.figure(figsize=(9,5))
        plt.plot(x_norm, yfp_trunc, color='orange')
       #plt.title(f"YFP - frame {index}")
        plt.xlabel("$$r_{norm}$$")
        plt.ylabel("$Intensity (a. u.)$")
        pdf_path = os.path.join(output_dir, f"{base}_YFP_{index}_normalized.pdf")
        plt.savefig(pdf_path)

        buf = BytesIO()
        plt.savefig(buf, format='png')
        buf.seek(0)
        gif_yfp_frames.append(imageio.imread(buf))
        buf.close()
        plt.close()

        plt.figure(figsize=(9,5))
        plt.plot(x_norm, mcherry_trunc, color='red', label='Matrix')
        plt.plot(x_norm, yfp_trunc, color='orange', label='Flagellated')
        #plt.title(f"Combined - frame {index}")
        plt.xlabel("$r_{norm}$")
        plt.ylabel("$Intensity \ (a. u.)$")
        plt.xlim(0, 1)
        plt.ylim(0, 110)
        plt.legend(loc='upper left', bbox_to_anchor=(1, 1), fontsize='small')
        pdf_path = os.path.join(output_dir, f"{base}_Combined_{index}_normalized.pdf")
        plt.savefig(pdf_path)

        buf = BytesIO()
        plt.savefig(buf, format='png')
        buf.seek(0)
        gif_combined_frames.append(imageio.imread(buf))
        buf.close()
        plt.close()

        plt.figure(figsize=(9,5))
        #plt.plot(x_norm, ratio, color='black', label='Matrix/Flagellated Ratio')
        plt.plot(x_norm, ratio_inv, color='black', label='FM-Ratio')
        #plt.title(f"FM-Ratio - frame {index}")
        plt.xlabel("$r_{norm}$")
        plt.ylabel("$Ratio \  (a. u.)$")
        plt.xlim(0, 1)
        plt.ylim(0, 5)
        plt.legend(loc='upper left', bbox_to_anchor=(1, 1), fontsize='small')
        pdf_path = os.path.join(output_dir, f"{base}_FM-Ratio_{index}_normalized.pdf")
        plt.savefig(pdf_path)       

        # Save to in-memory buffer for GIF
        buf = BytesIO()
        plt.savefig(buf, format='png')
        buf.seek(0)
        gif_combined_ratio_frames.append(imageio.imread(buf))
        buf.close()
        plt.close()


        

# Save GIFs for this dataset group
    if gif_mcherry_frames:
        imageio.mimsave(os.path.join(output_dir, f"{base}_mCherry_all_frames.gif"), gif_mcherry_frames, duration=0.115)

    if gif_yfp_frames:
        imageio.mimsave(os.path.join(output_dir, f"{base}_YFP_all_frames.gif"), gif_yfp_frames, duration=0.115)

    if gif_combined_frames:
        imageio.mimsave(os.path.join(output_dir, f"{base}_Combined_all_frames.gif"), gif_combined_frames, duration=0.115)

    if gif_combined_ratio_frames:
        imageio.mimsave(os.path.join(output_dir, f"{base}_Combined_Ratio_all_frames.gif"), gif_combined_ratio_frames, duration=0.115)

    if len(all_xnorm) == 0:
        print(f"No valid frames for {base}, skipping group plots.")
        continue

    print(f"  Creating group plots for {base}")

    cmap = cmc.batlow
    n = len(valid_indices)
    colors = [cmap(i / max(n-1, 1)) for i in range(n)]

    combined_indices = valid_indices[10::5]
    combined_xnorm = [all_xnorm[i] for i, idx in enumerate(valid_indices) if idx in combined_indices]
    combined_mcherry = [all_mcherry[i] for i, idx in enumerate(valid_indices) if idx in combined_indices]
    combined_yfp = [all_yfp[i] for i, idx in enumerate(valid_indices) if idx in combined_indices]
    combined_ratio = [all_ratio[i] for i, idx in enumerate(valid_indices) if idx in combined_indices]
    combined_ratio_inv = [all_ratio_inv[i] for i, idx in enumerate(valid_indices) if idx in combined_indices]
    combined_colors = [colors[i] for i, idx in enumerate(valid_indices) if idx in combined_indices]
    combined_labels = [combined_indices[i] for i in range(len(combined_indices))]

    plt.figure(figsize=(6,4))
    lines = []
    labels = []
    for xnorm, mc, yf, idx, col in zip(combined_xnorm, combined_mcherry, combined_yfp, combined_labels, combined_colors):
        l1, = plt.plot(xnorm, mc, color=col, linestyle='solid')
        l2, = plt.plot(xnorm, yf, color=col, linestyle='dashed')
        lines.extend([l1, l2])
        labels.extend([f"Frame {idx} – mCherry", f"Frame {idx} – YFP"])
    plt.title(f"{base} – mCherry + YFP (every 5th frame)")
    plt.xlabel("$r_{norm}$")
    plt.ylabel("Intensity (a. u.)")
    plt.legend(lines, labels, loc='upper left', bbox_to_anchor=(1, 1), fontsize='4')
    plt.savefig(os.path.join(output_dir, f"{base}_combined_every5_frames.pdf"))
    plt.close()

print("\nAll processing complete.")


# Plot the maximum(phi) and minimum (theta) of the FM-ratio as well as the position of the maxima

In [ ]:
import os
import re
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import LinearRegression
from scipy.signal import savgol_filter, find_peaks


plt.rcParams.update({'font.size': 30, 'font.family': 'Mathematica1', 'lines.linewidth': 4})

# parameters

slope_threshold = 7
slope_threshold_YFP = 150
min_norm = 0 #consider all peaks with a normalized radius above this value
max_norm = 0.98 #consider all peaks with a normalized radius below this value

smooth_window = 5 #Savitzky-Golay filter window for smoothing before peak detection
polyorder = 3 #Polynomial order for Savitzky-Golay filter
min_prominence = 0.001 #Minimum prominence for peak detection in the ratio curves


# Time mapping function

def frame_to_time(frame_idx):
    return 40 * frame_idx / 60


# Peak detection function with slope criteria for ratio curves

def find_valid_peak(values, xvals, norm_vals, slope_thr):
    values = values.reset_index(drop=True)
    xvals = xvals.reset_index(drop=True)
    norm_vals = norm_vals.reset_index(drop=True)

    if len(values) < smooth_window:
        return None

    smooth_values = savgol_filter(values.values, smooth_window, polyorder)
    smooth_series = pd.Series(smooth_values)

    strong_peaks, _ = find_peaks(smooth_series, prominence=min_prominence)
    all_peaks, _ = find_peaks(smooth_series)

    peak_positions = np.unique(np.concatenate((strong_peaks, all_peaks)))

    if smooth_series.iloc[-1] > smooth_series.iloc[-2]:
        peak_positions = np.append(peak_positions, len(smooth_series) - 1)

    if len(peak_positions) == 0:
        return None

    peak_indices = smooth_series.iloc[peak_positions].sort_values(ascending=False).index

    for idx in peak_indices:
        local_idx = idx

        left = max(0, local_idx - 3)
        right = min(len(smooth_series), local_idx + 4)

        if right - left < 5:
            continue

        y_window = smooth_series.iloc[left:right].values
        x_window = norm_vals.iloc[left:right].values.reshape(-1, 1)

        model = LinearRegression().fit(x_window, y_window)
        slope = model.coef_[0]

        if -slope_thr <= slope <= slope_thr:
            fitted_y = model.predict(x_window)
            real_norm = norm_vals.iloc[left:right].values

            return (
                smooth_series.iloc[idx],
                xvals.iloc[idx],
                norm_vals.iloc[idx],
                real_norm,
                fitted_y,
                x_window,
                y_window
            )

    return None


# Peak detection function with slope criteria for intensity curves
def find_global_minimum(values, xvals, norm_vals, slope_thr):
    values = values.reset_index(drop=True)
    xvals = xvals.reset_index(drop=True)
    norm_vals = norm_vals.reset_index(drop=True)

    if len(values) < smooth_window:
        return None

    smooth_values = savgol_filter(values.values, smooth_window, polyorder)
    smooth_series = pd.Series(smooth_values)

    inverted = -smooth_series
    minima_indices, _ = find_peaks(inverted, prominence=min_prominence)

    if len(minima_indices) == 0:
        return None

    sorted_minima = smooth_series.iloc[minima_indices].sort_values(ascending=True).index

    for idx in sorted_minima:
        left = max(0, idx - 3)
        right = min(len(smooth_series), idx + 4)

        if right - left < 5:
            continue

        y_window = smooth_series.iloc[left:right].values
        x_window = norm_vals.iloc[left:right].values.reshape(-1, 1)

        model = LinearRegression().fit(x_window, y_window)
        slope = model.coef_[0]

        if -slope_thr <= slope <= slope_thr:
            fitted_y = model.predict(x_window)
            real_norm = norm_vals.iloc[left:right].values

            return (
                smooth_series.iloc[idx],
                xvals.iloc[idx],
                norm_vals.iloc[idx],
                
            )

    global_idx = smooth_series.idxmin()

    left = max(0, global_idx - 3)
    right = min(len(smooth_series), global_idx + 4)

    if right - left >= 5:
        y_window = smooth_series.iloc[left:right].values
        x_window = norm_vals.iloc[left:right].values.reshape(-1, 1)

        model = LinearRegression().fit(x_window, y_window)
        fitted_y = model.predict(x_window)
        real_norm = norm_vals.iloc[left:right].values

        return (
            smooth_series.iloc[global_idx],
            xvals.iloc[global_idx],
            norm_vals.iloc[global_idx],
          
        )

    return None

# MAIN PROCESSING

def extract_timelapse_maxima(input_dir):
    pattern = re.compile(
        r"^(?P<base>.+)_(?P<channel>mCherry|YFP|Threshold)_(?P<index>\d+)\.csv$",
        re.IGNORECASE,
    )

    grouped = {}
    yfp_peakdata = {}
    mch_peakdata = {}

    for root, dirs, files in os.walk(input_dir):
        for f in files:
            match = pattern.match(f)
            if not match:
                continue

            base = match.group("base")
            channel = match.group("channel").capitalize()
            index = int(match.group("index"))
            full_path = os.path.join(root, f)

            grouped.setdefault(base, {})
            grouped[base].setdefault(index, {})
            grouped[base][index][channel] = full_path

    rows = []

    for base, frames in grouped.items():
        print(f"\nProcessing group: {base}")

        yfp_peakdata[base] = {}
        mch_peakdata[base] = {}

        for index, fdict in sorted(frames.items()):

            if index <= 1:
                continue

            if not all(k in fdict for k in ("Mcherry", "Yfp", "Threshold")):
                continue

            mcherry_df = pd.read_csv(fdict["Mcherry"]).dropna(how='all').reset_index(drop=True)
            yfp_df = pd.read_csv(fdict["Yfp"]).dropna(how='all').reset_index(drop=True)
            threshold_df = pd.read_csv(fdict["Threshold"]).dropna(how='all').reset_index(drop=True)

            if len(mcherry_df) != len(yfp_df):
                continue

            x = mcherry_df["Distance"]
            m = mcherry_df["Intensity"]
            y = yfp_df["Intensity"]
            thr = threshold_df["Intensity"]

            # Determine cutoff index based on threshold crossings
            idx_255 = thr[thr == 255].index

            if len(idx_255) == 0:
                cutoff_index = len(thr) - 1
            else:
                last_255_index = idx_255[-1]
                drops_after_255 = thr[
                    (thr < 15) & (thr.index > last_255_index)
                ].index

                if len(drops_after_255) > 0:
                    cutoff_index = drops_after_255[0]
                else:
                    cutoff_index = len(thr) - 1

            cutoff_index = max(0, min(cutoff_index, len(x) - 1))

            x_tr = x.iloc[:cutoff_index + 1]
            m_tr = m.iloc[:cutoff_index + 1]
            y_tr = y.iloc[:cutoff_index + 1]

            edge_distance = x_tr.iloc[-1]
            norm = x_tr / x_tr.iloc[-1]

            ratio = (m_tr / y_tr.replace(0, np.nan)).fillna(0)
            ratio2 = (y_tr / m_tr.replace(0, np.nan)).fillna(0)

            norm_radii = x_tr / edge_distance
            
            valid_indices = (norm_radii >= min_norm) & (norm_radii <= max_norm)

            if not valid_indices.any():
                continue

            ratio_valid = ratio[valid_indices]
            ratio2_valid = ratio2[valid_indices]

            x_valid = x_tr[valid_indices]
            norm_radii_valid = norm_radii[valid_indices]

            p1 = find_valid_peak(ratio_valid, x_valid, norm_radii_valid, slope_threshold)
            p2 = find_valid_peak(ratio2_valid, x_valid, norm_radii_valid, slope_threshold)
            pmin = find_global_minimum(ratio2_valid, x_valid, norm_radii_valid, slope_threshold)

            if p1 is None or p2 is None or pmin is None:
                continue

            max1, max1_r, max1_nr, _, _, _, _ = p1
            max2, max2_r, max2_nr, _, _, _, _ = p2
            min2, min2_r, min2_nr = pmin

            p_mch = find_valid_peak(m_tr, x_tr, norm_radii, slope_threshold_YFP)
            p_yfp = find_valid_peak(y_tr, x_tr, norm_radii, slope_threshold_YFP)

            if p_mch is None or p_yfp is None:
                continue

            max_mch, max_mch_r, max_mch_nr, real_norm_mch, fitted_mch, xwin_mch, ywin_mch = p_mch
            max_yfp, max_yfp_r, max_yfp_nr, real_norm_yfp, fitted_yfp, xwin_yfp, ywin_yfp = p_yfp

            yfp_peakdata[base][index] = {
                "x_tr": x_tr.values,
                "y_tr": y_tr.values,
                "peak_r": max_yfp_r,
                "peak_y": max_yfp,
                "norm_peak": max_yfp_nr,
                "norm_window": xwin_yfp.flatten(),
                "y_window": ywin_yfp,
                "fitted_y": fitted_yfp
            }

            mch_peakdata[base][index] = {
                "x_tr": x_tr.values,
                "y_tr": m_tr.values,
                "peak_r": max_mch_r,
                "peak_y": max_mch,
                "norm_peak": max_mch_nr,
                "norm_window": xwin_mch.flatten(),
                "y_window": ywin_mch,
                "fitted_y": fitted_mch
            }

            rows.append({
                "Base": base,
                "Frame": index,
                "Time_min": frame_to_time(index),
                "MaxRatio_MF": max1,
                "MinRatio_FM": min2,
                "MaxRatio_FM": max2,
                "NormPos_MF": min2_nr,
                "NormPos_FM": max2_nr,
                "ActualRadius": edge_distance,
                "Max_mCherry": max_mch,
                "Max_YFP": max_yfp,
                "NormPos_mCherry": max_mch_nr,
                "NormPos_YFP": max_yfp_nr,
                "RadPos_mCherry": max_mch_r,
                "RadPos_YFP": max_yfp_r,
                "norm_curve": norm.values,
                "ratio_curve_FM": ratio2.values,
                "ratio_curve_MF": ratio.values
            })

    return pd.DataFrame(rows), yfp_peakdata, mch_peakdata, input_dir


def plot_time_series(df, base, input_dir):
    output_dir = os.path.join(input_dir, "Plots/Data_Analysis", base)
    os.makedirs(output_dir, exist_ok=True)

    clean_base = base.replace("_RadialProfile_Angle", "")

    dfb = df[df["Base"] == base].sort_values("Time_min")

    plt.figure(figsize=(12,8))
    plt.plot(dfb["Time_min"], dfb["MaxRatio_MF"], "o")
    plt.xlabel("Time (h)")
    plt.ylabel("Max Matrix/Flagellated Ratio")
    plt.savefig(os.path.join(output_dir, f"{base}_Max_MatrixFlagellated_vs_time.pdf"))
    plt.show()

    plt.figure(figsize=(12,8))
    plt.plot(dfb["Time_min"], dfb["MaxRatio_FM"], "o", color="orange")
    plt.xlabel("Time (h)")
    plt.ylabel("$I_{FM,\\phi}$")
    plt.savefig(os.path.join(output_dir, f"{base}_Max_FlagellatedMatrix_vs_time.pdf"))
    plt.show()

    plt.figure(figsize=(12,8))
    plt.plot(dfb["Time_min"], dfb["NormPos_MF"], "o", color="blue")
    plt.xlabel("Time (h)")
    plt.ylabel("$r_{norm}$")
    plt.ylim(0, 1)
    plt.savefig(os.path.join(output_dir, f"{base}_Position_Max_MatrixFlagellated_vs_time.pdf"))
    plt.show()

    plt.figure(figsize=(12,8))
    plt.plot(dfb["Time_min"], dfb["NormPos_FM"], "o", color="red")
    plt.xlabel("Time (h)")
    plt.ylabel("$r_{norm}$")
    plt.ylim(0, 1)
    plt.savefig(os.path.join(output_dir, f"{base}_Position_Max_FlagellatedMatrix_vs_time.pdf"))
    plt.show()

    plt.figure(figsize=(12,8))
    plt.plot(dfb["Time_min"], dfb["MinRatio_FM"], "o", color="black", label="$I_{FM,\\theta}$")
    plt.plot(dfb["Time_min"], dfb["MaxRatio_FM"], "o", color="slategray", label="$I_{FM,\\phi}$")
    plt.xlabel("Time (h)")
    plt.ylabel("FM-Ratio")
    plt.legend()
    plt.ylim(0, 7)
    plt.xlim(0, 120)
    plt.savefig(os.path.join(output_dir, f"{base}_Min_Max_FM_vs_time.pdf"))
    plt.show()

    plt.figure(figsize=(12,8))
    plt.plot(dfb["Time_min"], dfb["NormPos_MF"], "o", color="black", label="$r_{\\theta}$ ")
    plt.plot(dfb["Time_min"], dfb["NormPos_FM"], "o", color="slategray", label="$r_{\\phi}$ ")
    plt.xlabel("Time (h)")
    plt.ylabel("$r_{norm}$")
    plt.legend()
    plt.ylim(0, 1)
    plt.xlim(0, 120)
    plt.savefig(os.path.join(output_dir, f"{base}_Position_Min_Max_FM_vs_time.pdf"))
    plt.show()

    plt.figure(figsize=(12,8))
    plt.plot(dfb["Time_min"], dfb["ActualRadius"], "o", color="green")
    plt.xlabel("Time (h)")
    plt.ylabel("$r$ (µm)")
    plt.title(f"{clean_base}: Colony Radius vs. Time")
    plt.savefig(os.path.join(output_dir, f"{base}_Max_Radius_vs_time.pdf"))
    plt.show()

    plt.figure(figsize=(12,8))
    plt.plot(dfb["Time_min"], dfb["Max_mCherry"], "o", color="magenta")
    plt.xlabel("Time (h)")
    plt.ylabel("$I_{m,\\beta}$ (a.u.)")
    plt.savefig(os.path.join(output_dir, f"{base}_Max_Matrix_vs_time.pdf"))
    plt.show()

    plt.figure(figsize=(12,8))
    plt.plot(dfb["Time_min"], dfb["Max_YFP"], "o", color="gold")
    plt.xlabel("Time (h)")
    plt.ylabel("$I_{Y,\\alpha}$ (a.u.)")
    plt.savefig(os.path.join(output_dir, f"{base}_Max_Flagellated_vs_time.pdf"))
    plt.show()

    plt.figure(figsize=(12,8))
    plt.plot(dfb["Time_min"], dfb["NormPos_mCherry"], "o", color="purple")
    plt.xlabel("Time (h)")
    plt.ylabel("$r_{norm}$")
    plt.ylim(0, 1)
    plt.savefig(os.path.join(output_dir, f"{base}_Position_Max_Matrix_vs_time.pdf"))
    plt.show()

    plt.figure(figsize=(6,4))
    plt.plot(dfb["Time_min"], dfb["NormPos_YFP"], "o", color="darkorange")
    plt.xlabel("Time (h)")
    plt.ylabel("$r_{norm}$")
    plt.ylim(0, 1)
    plt.savefig(os.path.join(output_dir, f"{base}_Position_Max_Flagellated_vs_time.pdf"))
    plt.show()

    plt.figure(figsize=(9,6))
    plt.plot(dfb["Time_min"], dfb["RadPos_mCherry"], "o", color="purple")
    plt.xlabel("Time (h)")
    plt.ylabel("Radius (µm)")
    plt.title(f"{clean_base}: Radius of max mCherry Intensity vs. Time")
    plt.savefig(os.path.join(output_dir, f"{base}_Radius_in_um_Max_Matrix_vs_time.pdf"))
    plt.show()

    plt.figure(figsize=(6,4))
    plt.plot(dfb["Time_min"], dfb["RadPos_YFP"], "o", color="darkorange")
    plt.xlabel("Time (h)")
    plt.ylabel("Radius (µm)")
    plt.title(f"{clean_base}: Radius of max YFP Intensity vs. Time")
    plt.savefig(os.path.join(output_dir, f"{base}_Radius_in_um_Max_Flagellated_vs_time.pdf"))
    plt.show()


def plot_ratio_profiles(df, base, input_dir):
    dfb = df[df["Base"] == base]

    for _, row in dfb.iterrows():

        norm_r = np.array(row["norm_curve"])
        ratio_curve_2 = np.array(row["ratio_curve_FM"])

        plt.figure(figsize=(6, 4))

        plt.plot(norm_r, ratio_curve_2,
                 '-', color='green', label="FM-ratio")

        plt.scatter(row["NormPos_FM"], row["MaxRatio_FM"],
                    color='black', zorder=10, label="Detected Max FM-ratio")

        plt.scatter(row["NormPos_MF"], row["MinRatio_FM"],
                    color='red', zorder=10, label="Detected Min FM-ratio")

        center = np.argmin(np.abs(norm_r - row["NormPos_FM"]))

        if 3 <= center < len(norm_r) - 3:
            y2_window = ratio_curve_2[center-3:center+4]
            x2_window = norm_r[center-3:center+4]

            valid = np.isfinite(y2_window)

            if np.sum(valid) >= 3:
                model2 = LinearRegression().fit(
                    x2_window[valid].reshape(-1, 1),
                    y2_window[valid]
                )
                fit_y2 = model2.predict(x2_window.reshape(-1, 1))

                plt.plot(x2_window, fit_y2,
                         '-', color='orange', linewidth=3,
                         label="Fitted Region FM")

        plt.xlabel("$r_{norm}$")
        plt.ylabel("Ratio")
        plt.xlim(0, 1)
        plt.title(f"{row['Base']} | t={row['Time_min']:.1f} h")

        plt.legend()
        plt.show()


# RUN

input_dir = input("Enter the timelapse root folder: ").strip()
df, yfp_peakdata, mch_peakdata, input_dir = extract_timelapse_maxima(input_dir)


if df is None or df.empty:
    print("No valid data.")
else:
    print(df.head())
    for base in df["Base"].unique():
        plot_time_series(df, base, input_dir)
        #plot_ratio_profiles(df, base, input_dir) #optional detailed ratio profile plots to check the reliability of the peak detection

### Ploting the FM-ratio integrals (psi) over time

In [ ]:
import os
import re
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import LinearRegression

plt.rcParams.update({'font.size': 28, 'lines.linewidth': 4})
# parameters

slope_threshold = 8 
slope_threshold_YFP = 150
min_norm = 0.1 # only points with normalized radius between 0.1 and 0.98 are considered for peak detection to avoid edge artifacts
max_norm = 0.98


# Time mapping function

def frame_to_time(frame_idx):

    if frame_idx <= 31:
        return 40 * frame_idx / 60
    else:
        return 40 * frame_idx / 60


# peak finder with smoothing

def find_valid_peak(values, xvals, norm_vals):

    candidate_indices = values.sort_values(ascending=False).index.tolist()

    for idx in candidate_indices:

        local_idx = values.index.get_loc(idx)

        if local_idx < 3 or local_idx > len(values) - 4:
            continue

        y_window = values.iloc[local_idx-3:local_idx+4].values
        x_window = norm_vals.iloc[local_idx-3:local_idx+4].values.reshape(-1,1)

        model = LinearRegression().fit(x_window, y_window)
        slope = model.coef_[0]

        if -slope_threshold <= slope <= slope_threshold:

            fitted_y = model.predict(x_window)
            real_norm = norm_vals.iloc[local_idx-3:local_idx+4].values

            return (
                values.loc[idx],
                xvals.loc[idx],
                norm_vals.loc[idx],
                real_norm,
                fitted_y,
                x_window,
                y_window
            )

    return None

# separate function for YFP with a different slope threshold
def find_valid_peak_YFP(values, xvals, norm_vals):

    candidate_indices = values.sort_values(ascending=False).index.tolist()

    for idx in candidate_indices:

        local_idx = values.index.get_loc(idx)

        if local_idx < 3 or local_idx > len(values) - 4:
            continue

        y_window = values.iloc[local_idx-3:local_idx+4].values
        x_window = norm_vals.iloc[local_idx-3:local_idx+4].values.reshape(-1,1)

        model = LinearRegression().fit(x_window, y_window)
        slope = model.coef_[0]

        if -slope_threshold_YFP <= slope <= slope_threshold_YFP:

            fitted_y = model.predict(x_window)
            real_norm = norm_vals.iloc[local_idx-3:local_idx+4].values

            return (
                values.loc[idx],
                xvals.loc[idx],
                norm_vals.loc[idx],
                real_norm,
                fitted_y,
                x_window,
                y_window
            )

    return None


# MAIN PROCESSING

def extract_timelapse_maxima(input_dir):

    pattern = re.compile(
        r"^(?P<base>.+)_(?P<channel>mCherry|YFP|Threshold)_(?P<index>\d+)\.csv$",
        re.IGNORECASE,
    )

    grouped = {}

    yfp_peakdata = {}
    mch_peakdata = {}

    for root, dirs, files in os.walk(input_dir):
        for f in files:

            match = pattern.match(f)
            if not match:
                continue

            base = match.group("base")
            channel = match.group("channel").capitalize()
            index = int(match.group("index"))
            full_path = os.path.join(root, f)

            grouped.setdefault(base, {})
            grouped[base].setdefault(index, {})
            grouped[base][index][channel] = full_path

    rows = []

    for base, frames in grouped.items():

        print(f"\nProcessing group: {base}")

        yfp_peakdata[base] = {}
        mch_peakdata[base] = {}

        for index, fdict in sorted(frames.items()):

            if index <= 1: #was 18 but i want to try 1 
                continue

            if not all(k in fdict for k in ("Mcherry", "Yfp", "Threshold")):
                print(f"Frame {index} missing channels.")
                continue

            mcherry_df = pd.read_csv(fdict["Mcherry"]).dropna(how='all').reset_index(drop=True)
            yfp_df = pd.read_csv(fdict["Yfp"]).dropna(how='all').reset_index(drop=True)
            threshold_df = pd.read_csv(fdict["Threshold"]).dropna(how='all').reset_index(drop=True)

            if len(mcherry_df) != len(yfp_df):
                continue

            x = mcherry_df["Distance"]
            m = mcherry_df["Intensity"]
            y = yfp_df["Intensity"]
            thr = threshold_df["Intensity"]

            threshold_y = thr

            idx_255 = threshold_y[threshold_y == 255].index

            if len(idx_255) == 0:

                cutoff_index = len(threshold_y) - 1

            else:

                last_255_index = idx_255[-1]

                drops_after_255 = threshold_y[
                    (threshold_y < 15) & (threshold_y.index > last_255_index)
                ].index

                if len(drops_after_255) > 0:
                    cutoff_index = drops_after_255[0]
                else:
                    cutoff_index = len(threshold_y) - 1

            cutoff_index = max(0, min(cutoff_index, len(x) - 1))

            x_tr = x.iloc[:cutoff_index + 1]
            m_tr = m.iloc[:cutoff_index + 1]
            y_tr = y.iloc[:cutoff_index + 1]

            edge_distance = x_tr.iloc[-1]

            norm = x_tr / x_tr.iloc[-1]

            """# Integrals over $r_{norm}$ 

            matrix_integral = np.trapz(m_tr.values, norm.values)
            yfp_integral = np.trapz(y_tr.values, norm.values)

            if matrix_integral == 0:
                integral_ratio = np.nan
            else:
                integral_ratio = yfp_integral / matrix_integral """
            # Integrals over actual radius with area weighting

            r_2 = x_tr.values
            I_matrix = m_tr.values
            I_yfp = y_tr.values

      
            dr_2 = np.gradient(r_2)
            # the integrals calculated as riemann sums
            matrix_integral = np.sum(I_matrix * r_2 * dr_2) / (np.pi * cutoff_index * cutoff_index) #normalize by area of the circle defined by the cutoff radius to get an average intensity per unit area
            yfp_integral = np.sum(I_yfp * r_2 * dr_2) / (np.pi * cutoff_index * cutoff_index)
            #matrix_integral = np.trapz(m_tr*norm_radii, norm_radii)
            #yfp_integral = np.trapz(y_tr*norm_radii, norm_radii)

            if matrix_integral == 0:
                integral_ratio = np.nan
            else:
                integral_ratio = yfp_integral / matrix_integral

            ratio = (m_tr / y_tr.replace(0, np.nan)).fillna(0)
            ratio2 = (y_tr / m_tr.replace(0, np.nan)).fillna(0)

            valid = (norm >= min_norm) & (norm <= max_norm)

            if not valid.any():
                continue

            p1 = find_valid_peak(ratio[valid], x_tr[valid], norm[valid])
            p2 = find_valid_peak(ratio2[valid], x_tr[valid], norm[valid])

            if p1 is None or p2 is None:
                continue

            max1, max1_r, max1_nr, _, _, _, _ = p1
            max2, max2_r, max2_nr, _, _, _, _ = p2

            p_mch = find_valid_peak_YFP(m_tr[valid], x_tr[valid], norm[valid])
            p_yfp = find_valid_peak_YFP(y_tr[valid], x_tr[valid], norm[valid])

            if p_mch is None or p_yfp is None:
                continue

            max_mch, max_mch_r, max_mch_nr, real_norm_mch, fitted_mch, xwin_mch, ywin_mch = p_mch
            max_yfp, max_yfp_r, max_yfp_nr, real_norm_yfp, fitted_yfp, xwin_yfp, ywin_yfp = p_yfp

            rows.append({
                "Base": base,
                "Frame": index,
                "Time_min": frame_to_time(index),
                "MaxRatio_MF": max1,
                "MaxRatio_FM": max2,
                "NormPos_MF": max1_nr,
                "NormPos_FM": max2_nr,
                "ActualRadius": edge_distance,
                "Max_mCherry": max_mch,
                "Max_YFP": max_yfp,
                "NormPos_mCherry": max_mch_nr,
                "NormPos_YFP": max_yfp_nr,
                "RadPos_mCherry": max_mch_r,
                "RadPos_YFP": max_yfp_r,
                "MatrixIntegral": matrix_integral,
                "YFPIntegral": yfp_integral,
                "IntegralRatio_YFP_Matrix": integral_ratio
            })

    return pd.DataFrame(rows), yfp_peakdata, mch_peakdata, input_dir


# Time series plotting function

def plot_time_series(df, base, input_dir):

    output_dir = os.path.join(input_dir, "Plots/Data_Analysis", base)
    os.makedirs(output_dir, exist_ok=True)

    clean_base = base.replace("_RadialProfile_Angle", "")

    dfb = df[df["Base"] == base].sort_values("Time_min")

    plt.figure(figsize=(12,8))
    plt.plot(dfb["Time_min"], dfb["MatrixIntegral"], "o", color="red")
    plt.xlabel("Time (h)")
    plt.ylabel("Matrix Integral")
    plt.savefig(os.path.join(output_dir, f"{base}_MatrixIntegral_vs_time.pdf"))
    plt.show()

    plt.figure(figsize=(12,8))
    plt.plot(dfb["Time_min"], dfb["YFPIntegral"], "o", color="orange")
    plt.xlabel("Time (h)")
    plt.ylabel("Flagellated Integral")
    plt.savefig(os.path.join(output_dir, f"{base}_FlagellatedIntegral_vs_time.pdf"))
    plt.show()

    plt.figure(figsize=(12,8))
    plt.plot(dfb["Time_min"], dfb["MatrixIntegral"], "o", color="red", label = "$\\int I_{m}$")
    plt.plot(dfb["Time_min"], dfb["YFPIntegral"], "o", color="orange", label = "$\\int I_{f}$")
    plt.xlabel("Time (h)")
    plt.ylabel("Flagellated Integral")
    plt.legend()
    plt.savefig(os.path.join(output_dir, f"{base}_FlagellatedandMatrixIntegral_vs_time.pdf"))
    plt.show()

    plt.figure(figsize=(12,8))
    plt.plot(dfb["Time_min"], dfb["IntegralRatio_YFP_Matrix"], "o", color="black")
    plt.xlabel("Time (h)")
    plt.ylabel("$\\psi$")
    plt.ylim(0.3, 2.5)
    plt.savefig(os.path.join(output_dir, f"{base}_IntegralRatio_vs_time.pdf"))
    plt.show()

    plt.figure(figsize=(12,8))
    plt.plot(dfb["ActualRadius"], dfb["IntegralRatio_YFP_Matrix"], "o", color="black")
    plt.xlabel("Colony Radius (µm)")
    plt.ylabel("$\\psi$")
    plt.ylim(0.3, 2.5)
    plt.savefig(os.path.join(output_dir, f"{base}_IntegralRatio_vs_radius.pdf"))
    plt.show()


# RUN

input_dir = input("Enter the timelapse root folder: ").strip()

df, yfp_peakdata, mch_peakdata, input_dir = extract_timelapse_maxima(input_dir)

if df is None or df.empty:
    print("No valid data.")
else:
    print(df.head())

    for base in df["Base"].unique():
        plot_time_series(df, base, input_dir)

### Detect the local maxima and plot them as a function of the radius. The time dimension is ilustrated in a colour scale

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from scipy.signal import savgol_filter
import cmcrameri.cm as cmc #for batlow colourmap


min_norm = 0 #only points with normalized radius between 0.1 and 0.98 are considered for peak detection to avoid edge artifacts
max_norm = 0.98
local_window = 0.15 #was 0.1
min_prominence = 0.005 #was 0.05
smooth_window = 7
polyorder = 3

# Peak detection function for local maxima in ratio curves with prominence and local window criteria
def find_all_local_peaks(values, xvals, norm_vals,
                         window=local_window,
                         min_prominence=min_prominence,
                         smooth_window=smooth_window,
                         polyorder=polyorder):

    if len(values) < smooth_window or smooth_window % 2 == 0:
        return []

    smooth_vals = savgol_filter(values.values, smooth_window, polyorder)
    peaks = []

    for i in range(1, len(values) - 1):

        if not (smooth_vals[i] > smooth_vals[i - 1] and
                smooth_vals[i] > smooth_vals[i + 1]):
            continue

        r0 = norm_vals.iloc[i]
        local = (norm_vals >= r0 - window) & (norm_vals <= r0 + window)
        local_vals = values[local]

        if values.iloc[i] != local_vals.max():
            continue

        baseline = np.median(local_vals)
        if values.iloc[i] - baseline < min_prominence:
            continue

        peaks.append((
            values.iloc[i],
            xvals.iloc[i],
            r0
        ))

    return peaks

# MAIN PROCESSING
input_dir = input("Enter timelapse root folder: ").strip()
if not os.path.isdir(input_dir):
    raise SystemExit("Invalid directory")

pattern = re.compile(
    r"^(?P<base>.+)_(?P<channel>mCherry|YFP|Threshold)_(?P<index>\d+)\.csv$",
    re.IGNORECASE
)

grouped = {}

for root, _, files in os.walk(input_dir):
    for f in files:
        m = pattern.match(f)
        if not m:
            continue

        base = m.group("base")
        channel = m.group("channel").capitalize()
        idx = int(m.group("index"))

        grouped.setdefault(base, {}).setdefault(idx, {})[channel] = os.path.join(root, f)


for base, frames in grouped.items():

    data = {
        "ratio_mf": {"frame": [], "norm": [], "x": [], "intensity": []},
        "ratio_fm": {"frame": [], "norm": [], "x": [], "intensity": []},
        "yfp":      {"frame": [], "norm": [], "x": [], "intensity": []},
        "mch":      {"frame": [], "norm": [], "x": [], "intensity": []},
    }

    frames_used = []

    for idx in sorted(frames):

        if idx <= 10:
            continue    

        fdict = frames[idx]
        if not all(k in fdict for k in ("Mcherry", "Yfp", "Threshold")):
            continue

        mch = pd.read_csv(fdict["Mcherry"])
        yfp = pd.read_csv(fdict["Yfp"])
        thr = pd.read_csv(fdict["Threshold"])

        if len(mch) != len(yfp):
            continue

        x = mch["Distance"]
        m = mch["Intensity"]
        y = yfp["Intensity"]
        t = thr["Intensity"]


        #edge detection 
        threshold_y = t

        # Find last index where mask is fully present (255)
        idx_255 = threshold_y[threshold_y == 255].index

        if len(idx_255) == 0:
            
            cutoff_index = len(threshold_y) - 1
        else:
            last_255_index = idx_255[-1]

            # First drop below threshold AFTER last solid mask
            drops_after_255 = threshold_y[
                (threshold_y < 15) & (threshold_y.index > last_255_index)
            ].index

            if len(drops_after_255) > 0:
                cutoff_index = drops_after_255[0]
            else:
                cutoff_index = len(threshold_y) - 1

        # Clamp cutoff safely
        cutoff_index = max(0, min(cutoff_index, len(x) - 1))

        # Trim profiles to colony edge
        x_tr = x.iloc[:cutoff_index + 1]
        m_tr = m.iloc[:cutoff_index + 1]
        y_tr = y.iloc[:cutoff_index + 1]

        # Actual colony radius
        edge_distance = x_tr.iloc[-1]

        if len(x_tr) < smooth_window:
            continue

        norm = x_tr / x_tr.iloc[-1]
        valid = (norm >= min_norm) & (norm <= max_norm)
        if not valid.any():
            continue

        ratio_mf = (m_tr / y_tr.replace(0, np.nan)).fillna(0)
        ratio_fm = (y_tr / m_tr.replace(0, np.nan)).fillna(0)

        peak_sets = {
            "ratio_mf": find_all_local_peaks(ratio_mf[valid], x_tr[valid], norm[valid]),
            "ratio_fm": find_all_local_peaks(ratio_fm[valid], x_tr[valid], norm[valid]),
            "yfp":      find_all_local_peaks(y_tr[valid],      x_tr[valid], norm[valid]),
            "mch":      find_all_local_peaks(m_tr[valid],      x_tr[valid], norm[valid]),
        }

        for key, peaks in peak_sets.items():
            for inten, xval, r in peaks:
                data[key]["frame"].append(idx)
                data[key]["norm"].append(r)
                data[key]["x"].append(xval)
                data[key]["intensity"].append(inten)
                frames_used.append(idx)

    if not frames_used:
        print(f"No peaks for {base}")
        continue

    outdir = os.path.join(input_dir, "Plots", base)
    os.makedirs(outdir, exist_ok=True)

    unique_frames = sorted(set(frames_used))
    cmap = cmc.batlow #for batlow colourmap

    #cmap = plt.cm.viridis #original colormap
    frame_color = {f: cmap(i / max(len(unique_frames) - 1, 1))
                   for i, f in enumerate(unique_frames)}

    def plot_peaks(data_dict, xkey, xlabel, xlim, ylabel, filename, title):
        plt.figure(figsize=(6, 4))
        for f, xval, inten in zip(
            data_dict["frame"],
            data_dict[xkey],
            data_dict["intensity"]
        ):
            plt.scatter(xval, inten, color=frame_color[f],
                        s=30, edgecolor="k")

        sm = plt.cm.ScalarMappable(
            cmap=cmap,
            norm=plt.Normalize(min(unique_frames), max(unique_frames))
        )
        sm.set_array([])
        plt.colorbar(sm, label="Frame")

        plt.xlabel(xlabel)
        plt.ylabel(ylabel)
        plt.xlim(xlim)
        plt.ylim(0, None)

        plt.tight_layout()
        plt.savefig(os.path.join(outdir, filename))
        plt.close()

    plot_defs = [
        ("ratio_mf", "Ratio (Matrix / Flagellated)"),
        ("ratio_fm", "Ratio (Flagellated / Matrix)"),
        ("yfp", "Intensity (Flagellated)"),
        ("mch", "Intensity (Matrix)")
    ]

    for key, ylabel in plot_defs:
        plot_peaks(
            data[key], "norm",
            "$r_{norm}$", (0, 1),
            ylabel,
            f"{base}_{key}_norm.pdf",
            f"{base} – {ylabel} ($r_{norm}$)"
        )

        plot_peaks(
            data[key], "x",
            "Radius ($\mu m$)", None,
            ylabel,
            f"{base}_{key}_raw.pdf",
            f"{base} – {ylabel} (raw radius)"
        )

    print(f"Plotted all local maxima for {base} → {outdir}")
